# Ces données ne sont pas concluantes, il s'agit d'un partage informatif et il faudra encore pas mal trafficoter le réseau pour essayer de voir si on peut obtenir un truc correct. => Pooling pour éviter d'avoir autant de pixels 128 x 128 que le réseau peu apprendre par coeur

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import json
import time

# Our own package: the same data pipeline and training functions for every model
from museum_guide.data import prepare_subset
from museum_guide.training import count_parameters, evaluate, fit

# Fix the random seed so that every run gives the same results
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load, split, standardise and batch the genre subset (takes about a minute)
genre = prepare_subset("genre")
print("device:", device, "| classes:", len(genre["classes"]), "| train batches:", len(genre["train_loader"]))

# Deliverable 1: a feed-forward network for artwork classification

## Design choices

| Element | Choice | Why |
|---|---|---|
| Input | 3 x 128 x 128 image, flattened to 49 152 values | default size of the course script (`--size 128`) |
| Hidden layers | 2 layers, 256 then 128 neurons | a simple network, close to the course examples |
| Activation | ReLU | default for hidden layers (TP05) |
| Initialisation | He | recommended with ReLU (TP05, Part D) |
| Output and loss | one logit per class + `CrossEntropyLoss` | standard multi-class recipe (TP05, TP06) |
| Optimiser | Adam, learning rate 1e-3 | usual default of Adam (TP06) |
| Training | 30 epochs, mini-batches of 64 | long enough to see overfitting appear (TP06) |
| Regularisation | none | added and measured in deliverable 2 |

In [ ]:
def make_mlp(n_classes, image_shape=(3, 128, 128), hidden_sizes=(256, 128)):
    """Build a feed-forward network (MLP) for colour images, as in TP05 and TP06.

    Args:
        n_classes: number of classes to predict (11 genres or 11 genres).
        image_shape: shape of one input image, (channels, height, width).
        hidden_sizes: number of neurons of each hidden layer.

    Returns:
        An nn.Sequential that maps a batch (B, 3, 128, 128) to (B, n_classes) logits.
    """
    # Turn each image into a vector: (B, 3, 128, 128) -> (B, 49152)
    layers = [nn.Flatten()]
    n_in = image_shape[0] * image_shape[1] * image_shape[2]
    for n_out in hidden_sizes:
        # One hidden layer followed by a ReLU (TP05, Part C)
        layers += [nn.Linear(n_in, n_out), nn.ReLU()]
        n_in = n_out
    # Output layer: one score (logit) per class, no softmax since CrossEntropyLoss applies it
    layers.append(nn.Linear(n_in, n_classes))
    model = nn.Sequential(*layers)

    # He initialisation, recommended for ReLU networks (TP05 Part D, TP06 exercise 21)
    for layer in model.modules():
        if isinstance(layer, nn.Linear):
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)
    return model


torch.manual_seed(0)
model = make_mlp(n_classes=len(genre["classes"])).to(device)
print(model)
print("parameters:", count_parameters(model))

# Check the output shape on one mini-batch
xb, yb = next(iter(genre["train_loader"]))
print("output:", tuple(model(xb.to(device)).shape))

In [ ]:
# Start training 

N_EPOCHS = 30
LEARNING_RATE = 1e-3

# Start again from a fresh network, so that the run does not depend on the cells above
torch.manual_seed(0)
genre_model = make_mlp(n_classes=len(genre["classes"])).to(device)

# The loss and the optimiser chosen together (see the design choices table)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(genre_model.parameters(), lr=LEARNING_RATE)

# Train for N_EPOCHS and keep the history of the metrics for the plots
genre_history = fit(
    genre_model, genre["train_loader"], genre["val_loader"], loss_fn, optimizer, N_EPOCHS, device=device
)

In [ ]:
genre_history["val_acc"]

In [ ]:
def plot_history(history, title=""):
    """Plot the loss and accuracy curves of a training run, train versus validation, as in TP06.

    Args:
        history: the dictionary returned by fit.
        title: title of the figure.
    """
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

    # Left: the loss (lower is better)
    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="validation")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("cross-entropy loss")
    axes[0].legend()

    # Right: the accuracy (higher is better)
    axes[1].plot(epochs, history["train_acc"], label="train")
    axes[1].plot(epochs, history["val_acc"], label="validation")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("accuracy")
    axes[1].legend()

    fig.suptitle(title)
    plt.show()


plot_history(genre_history, "Feed-forward network, 11 genres")

## Try 2: a simpler network (one hidden layer of 128 neurons)

The first network (256 then 128 neurons, 12.6 M weights) learnt the training images by heart:
the validation loss rose from epoch 2 on, and the validation accuracy stayed close to chance (about 4 %).

| Element | Try 1 | Try 2 |
|---|---|---|
| Hidden layers | 256 then 128 neurons | **1 layer of 128 neurons** |
| Weights | 12.6 M | **6.3 M** |
| Learning rate | 1e-3 | 1e-3 |
| Everything else | ReLU, He, Adam, 30 epochs | unchanged |

In [ ]:
# Second try: a simpler network with one hidden layer of 128 neurons (option A),
# because the 256-128 network learnt the training images by heart
torch.manual_seed(0)
genre_model_a = make_mlp(n_classes=len(genre["classes"]), hidden_sizes=(128,)).to(device)
print("parameters:", count_parameters(genre_model_a))

# Same loss and same optimiser settings as the first try, only the network changes
optimizer = torch.optim.Adam(genre_model_a.parameters(), lr=LEARNING_RATE)
genre_history_a = fit(
    genre_model_a, genre["train_loader"], genre["val_loader"], loss_fn, optimizer, N_EPOCHS, device=device
)
plot_history(genre_history_a, "Feed-forward network 49152-128-11, 11 genres")

## Try 3: a basic network (one hidden layer of 32 neurons)

Try 2 still learnt by heart, and its loss reached 20 to 30 with jumps on the training curve:
the network gives very confident wrong answers. We first test whether a much smaller network is enough.

| Element | Try 2 | Try 3 |
|---|---|---|
| Hidden layers | 1 layer of 128 neurons | **1 layer of 32 neurons** |
| Weights | 6.3 M | **1.6 M** |
| Learning rate | 1e-3 | 1e-3 |
| Everything else | ReLU, He, Adam, 30 epochs | unchanged |

In [ ]:
# Third try: a basic network with one hidden layer of 32 neurons.
# Only the size changes: the learning rate stays at 1e-3.
torch.manual_seed(0)
genre_model_c = make_mlp(n_classes=len(genre["classes"]), hidden_sizes=(32,)).to(device)
print("parameters:", count_parameters(genre_model_c))

optimizer = torch.optim.Adam(genre_model_c.parameters(), lr=1e-3)
genre_history_c = fit(
    genre_model_c, genre["train_loader"], genre["val_loader"], loss_fn, optimizer, N_EPOCHS, device=device
)
plot_history(genre_history_c, "Feed-forward network 49152-32-11, learning rate 1e-3")

## Tries 4 to 6: the same three networks with a learning rate 10 times smaller

Tries 1 to 3 changed the size of the network and none of them generalised: the validation accuracy
stayed close to chance (about 4 %), and the losses were very high with jumps on the training curves.
Hypothesis: the steps of the optimiser are too large. Adam moves each weight by about the learning rate
at every step, and each neuron of the first layer has 49 152 inputs: all these small moves add up to
large jumps. We train the same three networks again with a learning rate of 1e-4 instead of 1e-3,
and change nothing else.

In [ ]:
def train_genre_mlp(hidden_sizes, learning_rate, n_epochs=N_EPOCHS):
    """Build a fresh feed-forward network for the genres and train it with Adam.

    Args:
        hidden_sizes: number of neurons of each hidden layer, for example (128,).
        learning_rate: learning rate of Adam.
        n_epochs: number of epochs.

    Returns:
        (model, history): the trained network and the history returned by fit.
    """
    # Same seed for every try: same initial weights and same order of the mini-batches
    torch.manual_seed(0)
    model = make_mlp(n_classes=len(genre["classes"]), hidden_sizes=hidden_sizes).to(device)
    print("parameters:", count_parameters(model))

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = fit(
        model, genre["train_loader"], genre["val_loader"], loss_fn, optimizer, n_epochs, device=device
    )
    return model, history

### Try 4: two hidden layers (256 then 128 neurons), learning rate 1e-4

| Element | Try 1 | Try 4 |
|---|---|---|
| Hidden layers | 256 then 128 neurons | 256 then 128 neurons |
| Weights | 12.6 M | 12.6 M |
| Learning rate | 1e-3 | **1e-4** |
| Everything else | ReLU, He, Adam, 30 epochs | unchanged |

In [ ]:
# Try 4: the network of try 1 with a learning rate 10 times smaller
genre_model_4, genre_history_4 = train_genre_mlp(hidden_sizes=(256, 128), learning_rate=1e-4)
plot_history(genre_history_4, "Try 4: 49152-256-128-11, learning rate 1e-4")

### Try 5: one hidden layer of 128 neurons, learning rate 1e-4

| Element | Try 2 | Try 5 |
|---|---|---|
| Hidden layers | 1 layer of 128 neurons | 1 layer of 128 neurons |
| Weights | 6.3 M | 6.3 M |
| Learning rate | 1e-3 | **1e-4** |
| Everything else | ReLU, He, Adam, 30 epochs | unchanged |

In [ ]:
# Try 5: the network of try 2 with a learning rate 10 times smaller
genre_model_5, genre_history_5 = train_genre_mlp(hidden_sizes=(128,), learning_rate=1e-4)
plot_history(genre_history_5, "Try 5: 49152-128-11, learning rate 1e-4")

### Try 6: one hidden layer of 32 neurons, learning rate 1e-4

| Element | Try 3 | Try 6 |
|---|---|---|
| Hidden layers | 1 layer of 32 neurons | 1 layer of 32 neurons |
| Weights | 1.6 M | 1.6 M |
| Learning rate | 1e-3 | **1e-4** |
| Everything else | ReLU, He, Adam, 30 epochs | unchanged |

In [ ]:
# Try 6: the network of try 3 with a learning rate 10 times smaller
genre_model_6, genre_history_6 = train_genre_mlp(hidden_sizes=(32,), learning_rate=1e-4)
plot_history(genre_history_6, "Try 6: 49152-32-11, learning rate 1e-4")


## Tries 7 and 8: average pooling before the dense network

All the previous networks flatten the 49 152 pixel values: too many inputs for about 5 200
training images, and the dense layer does not know which pixels are neighbours (course S07, slide 4).
We keep the network of try 4 (256 then 128 neurons, learning rate 1e-4) and add one layer in front of it:
`nn.AvgPool2d(k)` replaces each block of k x k pixels by its mean (course S07, slide 12).
It has no weight to learn, so the model stays a dense network, with k ** 2 times fewer inputs.

| Element | Try 4 | Try 7 | Try 8 |
|---|---|---|---|
| Pooling before the network | none | **4 x 4 average** | **8 x 8 average** |
| Image seen by the dense layers | 128 x 128 | 32 x 32 | 16 x 16 |
| Inputs | 49 152 | 3 072 | 768 |
| Everything else | 256-128, ReLU, He, Adam lr 1e-4, 30 epochs | unchanged | unchanged |

In [ ]:
def make_pooled_mlp(n_classes, pool_size, hidden_sizes=(256, 128)):
    """Build an average pooling layer followed by the feed-forward network of make_mlp.

    Args:
        n_classes: number of classes to predict (11 genres or 11 genres).
        pool_size: each block of pool_size x pool_size pixels is replaced by its mean.
        hidden_sizes: number of neurons of each hidden layer of the dense network.

    Returns:
        An nn.Sequential that maps a batch (B, 3, 128, 128) to (B, n_classes) logits.
    """
    # Size of the image after pooling, for example 128 // 4 = 32
    pooled_side = 128 // pool_size
    return nn.Sequential(
        # Summarise each block of pixels by its mean (course S07, slide 12): no weight to learn
        nn.AvgPool2d(pool_size),
        # The unchanged dense network, built for the smaller pooled image
        make_mlp(n_classes, image_shape=(3, pooled_side, pooled_side), hidden_sizes=hidden_sizes),
    )


def train_genre_pooled_mlp(pool_size, learning_rate=1e-4, hidden_sizes=(256, 128), n_epochs=N_EPOCHS):
    """Build a fresh pooled network for the genres and train it with Adam, like train_genre_mlp.

    Returns:
        (model, history): the trained network and the history returned by fit.
    """
    # Same seed as the other tries: same initial weights and same order of the mini-batches
    torch.manual_seed(0)
    model = make_pooled_mlp(len(genre["classes"]), pool_size, hidden_sizes).to(device)
    print("parameters:", count_parameters(model))

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = fit(
        model, genre["train_loader"], genre["val_loader"], loss_fn, optimizer, n_epochs, device=device
    )
    return model, history


# Quick check of the number of weights, before any training
for pool in [4, 8]:
    print(f"pool {pool}: {count_parameters(make_pooled_mlp(len(genre['classes']), pool)):,} parameters")

In [ ]:
# Try 7: 4 x 4 average pooling (the network sees a 32 x 32 image), then the network of try 4
genre_model_7, genre_history_7 = train_genre_pooled_mlp(pool_size=4)
plot_history(genre_history_7, "Try 7: AvgPool 4 + 3072-256-128-11, learning rate 1e-4")

In [ ]:
# Try 8: 8 x 8 average pooling (the network sees a 16 x 16 image), then the network of try 4
genre_model_8, genre_history_8 = train_genre_pooled_mlp(pool_size=8)
plot_history(genre_history_8, "Try 8: AvgPool 8 + 768-256-128-11, learning rate 1e-4")

## Summary of the eight tries

For each try: the epoch with the best validation accuracy, and that accuracy.
Chance level with 11 genres: about 0.09.

In [ ]:
# One line per try: (number, hidden layers, learning rate, pooling, history returned by fit).
# The history variables are the ones created in the cells above.
tries = [
    (1, "2 layers: 256, 128", 1e-3, "none", genre_history),
    (2, "1 layer: 128", 1e-3, "none", genre_history_a),
    (3, "1 layer: 32", 1e-3, "none", genre_history_c),
    (4, "2 layers: 256, 128", 1e-4, "none", genre_history_4),
    (5, "1 layer: 128", 1e-4, "none", genre_history_5),
    (6, "1 layer: 32", 1e-4, "none", genre_history_6),
    (7, "2 layers: 256, 128", 1e-4, "4 x 4 average", genre_history_7),
    (8, "2 layers: 256, 128", 1e-4, "8 x 8 average", genre_history_8),
]

rows = []
for number, layers, learning_rate, pooling, history in tries:
    # Best validation accuracy over the 30 epochs, and the epoch where it was reached (counted from 1)
    best_acc = max(history["val_acc"])
    best_epoch = history["val_acc"].index(best_acc) + 1
    rows.append({
        "try": number,
        "hidden layers": layers,
        "learning rate": learning_rate,
        "pooling": pooling,
        "best epoch": best_epoch,
        "best val accuracy": round(best_acc, 3),
    })

# One row per try, indexed by the try number
genre_summary = pd.DataFrame(rows).set_index("try")
genre_summary.sort_values("best val accuracy", ascending=False)

## Final network: try 7 trained for 12 epochs

Try 7 (4 x 4 average pooling, then 256-128, learning rate 1e-4) has the best validation accuracy
of the eight tries, reached at epoch 12. The number of epochs is a setting chosen on validation,
like the learning rate: we train the same network again for 12 epochs instead of 30.
With the same seed, these 12 epochs are the same as the first 12 epochs of try 7.

In [ ]:
FINAL_EPOCHS = 12  # epoch of the best validation accuracy of try 7 (see the summary table)

# Same settings as try 7, only the number of epochs changes
final_model, final_history = train_genre_pooled_mlp(pool_size=4, n_epochs=FINAL_EPOCHS)

# Check: same seed, so the last validation accuracy must equal the one of try 7 at epoch 12
print(
    f"validation accuracy at epoch {FINAL_EPOCHS}: {final_history['val_acc'][-1]:.3f} "
    f"(try 7 at the same epoch: {genre_history_7['val_acc'][FINAL_EPOCHS - 1]:.3f})"
)
plot_history(final_history, f"Final network: AvgPool 4 + 3072-256-128-11, {FINAL_EPOCHS} epochs")

## Final evaluation on the test set (once)

The final network is now measured once on the test images, never used for training
nor for choosing the model or the number of epochs.

In [ ]:
# Folder of the team's figures (created if it does not exist yet)
FIG_DIR = Path("..") / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# The final network, measured ONCE on the test images
test_loss, test_acc = evaluate(final_model, genre["test_loader"], loss_fn, device=device)
n_classes = len(genre["classes"])
print(
    f"TEST accuracy = {test_acc:.3f} "
    f"| validation accuracy {final_history['val_acc'][-1]:.3f} | chance {1 / n_classes:.3f}"
)

# Predicted class of every test image (the test loader is not shuffled, so the order is kept)
final_model.eval()
with torch.no_grad():
    y_pred = torch.cat(
        [final_model(xb.to(device)).argmax(dim=1).cpu() for xb, _ in genre["test_loader"]]
    ).numpy()

# True class of the same test images, in the same order
y_true = genre["y"][torch.as_tensor(genre["test_idx"])].numpy()

## Confusion matrix

In [ ]:
class_names = genre["classes"]

# Entry [i, j] = number of test images of true class i predicted as class j (TP05, exercise 7)
cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))

fig, ax = plt.subplots(figsize=(14, 12))  # 11 classes: a large figure so that the numbers stay readable
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, xticks_rotation=90, colorbar=False, cmap="Blues"
)
ax.set_title(f"Feed-forward network, 11 genres · test accuracy {test_acc:.2f}")
fig.tight_layout()
fig.savefig(FIG_DIR / "mlp_genre_confusion_test.png", dpi=150)  # saved for the slides
plt.show()

## Accuracy per genre and most frequent error

In [ ]:
# Accuracy of each genre: correct predictions divided by the number of test images of that genre
per_class = cm.diagonal() / cm.sum(axis=1)
for name, acc in sorted(zip(class_names, per_class), key=lambda pair: -pair[1]):  # best genres first
    print(f"  {name:28s} {acc:.2f}")

# Most frequent mistake: the largest entry outside the diagonal (TP05, exercise 7)
off_diagonal = cm.copy()
np.fill_diagonal(off_diagonal, 0)
i, j = np.unravel_index(off_diagonal.argmax(), off_diagonal.shape)
print(f"most frequent mistake: true '{class_names[i]}' predicted as '{class_names[j]}' ({off_diagonal[i, j]} times)")